# 得到reg_year_data_short_long_two_ponit.csv

In [ ]:
file1 = "E:\\大四科研\\其他科研\\一带一路_合并\\rank\\reg_file_merge_beta.csv"
file2 = "E:\\大四科研\\其他科研\\一带一路_合并\\rank\\reg_year_data_201507_201606.csv"

df1 = pd.read_csv(file1)
columns_to_scale = ['ch4mkt_beta', 'ch4smb_beta', 'ch4vmg_beta', 'ch4pmo_beta']
df1[columns_to_scale] = df1[columns_to_scale] * 100
### df1.to_csv(r"E:\大四科研\其他科研\一带一路_合并\rank\control_beta_month.csv",index = False)

df1['year'] = df1['Trdmnt'].astype(str).str.slice(0, 4).astype(int)  
df1['month'] = df1['Trdmnt'].astype(str).str.slice(4, 6)  
df1 = df1[df1['month'] == '06']
df1 = df1.rename(columns={'Stkcd': 'asset'})
df1 = df1.drop(columns=['Trdmnt', 'month','xret'])


df2 = pd.read_csv(file2)
df2 = df2.rename(columns={'date': 'year'})
df2 = df2[['asset','year','br','compounded_xret']]

merged_df = pd.merge(df1, df2, on=['asset', 'year'], how='inner')

df3 = df2[['asset', 'year', 'compounded_xret']].copy()
df3 = df3.rename(columns={'compounded_xret': 'compounded_xret_beta'})
df3['year'] = df3['year'].astype(int) + 1

final_df = pd.merge(merged_df, df3, on=['asset', 'year'], how='inner')

final_df = final_df[final_df['br'] != 0]

final_df = final_df.dropna()
final_df['rank_beta'] = final_df.groupby('year')['beta_bri'].transform(lambda x: 2 * (x.rank(method='min') - 1) / (len(x) - 1) - 1)
final_df['rank_br'] = final_df.groupby('year')['br'].transform(lambda x: 2 * (x.rank(method='min') - 1) / (len(x) - 1) - 1)

final_df.to_csv(r"E:\大四科研\其他科研\一带一路_合并\rank\reg_year_data_short_long_two_ponit.csv",index = False)

# 得到reg_year_data_short_long_two_ponit1118.csv

In [ ]:
import pandas as pd

# 文件路径
control_beta_path = r'C:\Users\廖金升\Desktop\毕业论文重开\data\input\control_beta_month.csv'
reg_year_data_path = r'C:\Users\廖金升\Desktop\毕业论文重开\data\input\reg_year_data_short_long_two_ponit.csv'
output_path = r'C:\Users\廖金升\Desktop\毕业论文重开\data\input\reg_year_data_short_long_two_ponit1118.csv'

# 读取 control_beta_month.csv
control_beta_df = pd.read_csv(control_beta_path)

# 提取年份和月份
control_beta_df['year'] = control_beta_df['date'] // 100  # 提取年份
control_beta_df['month'] = control_beta_df['date'] % 100  # 提取月份

# 只保留每年的6月份数据
june_beta_df = control_beta_df[control_beta_df['month'] == 6].copy()

# 将年份减 1
june_beta_df['year'] -= 1

# 准备合并数据的四因子 beta，格式化年份
shifted_beta_df = june_beta_df[['asset', 'year', 'ch4mkt_beta', 'ch4smb_beta', 'ch4vmg_beta', 'ch4pmo_beta']]
shifted_beta_df.rename(columns={'year': 'date'}, inplace=True)  # 调整列名为 date（年份格式）

# 读取 reg_year_data_short_long_two_ponit.csv
reg_year_data_df = pd.read_csv(reg_year_data_path)

# 删除原始的四因子 beta 列
columns_to_remove = ['ch4mkt_beta', 'ch4smb_beta', 'ch4vmg_beta', 'ch4pmo_beta']
reg_year_data_df = reg_year_data_df.drop(columns=columns_to_remove, errors='ignore')

# 合并处理后的四因子 beta 数据
merged_df = pd.merge(reg_year_data_df, shifted_beta_df, on=['asset', 'date'], how='inner')

# 保存结果为 reg_year_data_short_long_two_ponit1118.csv
merged_df.to_csv(output_path, index=False)

print(f"处理完成，结果已保存至 {output_path}")